In [83]:
from qdrant_client import QdrantClient, models
from langchain.text_splitter import CharacterTextSplitter
import requests

def setup_database(localhost, port):        
    if not requests.get(f'http://{localhost}:{port}'):
        raise Exception(f'Qdrant server is not running at http://{localhost}:{port}')
        
    client = QdrantClient(location=localhost, port=port)

    return client

client = setup_database(localhost='0.0.0.0', port=6333)

In [84]:
collection_name = "it_area"

count = client.count(collection_name=collection_name).count
print(f"Количество векторов в коллекции '{collection_name}': {count}")

Количество векторов в коллекции 'it_area': 40


In [85]:
import re

with open('/Users/richardgurtsiev/Desktop/projects/save/delete_2024/del/dl_skip/advisor/agent/examples.txt') as f:
    raw_text = f.read()


def get_chunks(text):
    normalized_text = re.sub(r'-{40,}', '-' * 40, text)
    text_splitter = CharacterTextSplitter(
        separator="-" * 40,
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len
    )

    chunks = text_splitter.split_text(normalized_text)
    return chunks

texts = get_chunks(raw_text)

Created a chunk of size 1089, which is longer than the specified 1000
Created a chunk of size 1164, which is longer than the specified 1000


In [86]:
len(embedding)

NameError: name 'embedding' is not defined

# Embeddings

In [68]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("deepvk/USER-bge-m3",)
embedding = model.encode(texts[0], normalize_embeddings=True)

In [60]:
client.collection_exists(collection_name='it_area')

True

In [77]:
from tqdm import tqdm
def upload_database(collection_name: str=None, chunks: list=None):
    for idx, chunk in tqdm(enumerate(chunks)):
        embedding = model.encode(chunk, normalize_embeddings=True)

        client.upsert(
            collection_name=collection_name,
            points=[
                models.PointStruct(
                    id=idx,
                    vector=embedding[idx],
                    payload={
                        "content": chunk,
                        "content": row["content_changed"],
                        "category": row["category"],
                        "catalog": row["catalog"]
                    }
                )
            ]
        )


    
    

def create_database(collection_name: str=None, embedding_size: int=None):
    if client.collection_exists(collection_name=collection_name):
        return
    client.create_collection(
        collection_name=collection_name,
            vectors_config=models.VectorParams(
            size=embedding_size,
            distance=models.Distance.COSINE
        )
    )

# create_database(collection_name='it_area', embedding_size=1024)
upload_database(collection_name='it_area', chunks=texts)


3it [00:00, 16131.94it/s]

Имя вакансии: DevOps (Middle+ / Senior)
Опыт работы: От 3 до 6 лет
Описание: Компания использует технологии: CI/CD (TeamCity, Jenkins), мониторинг (Zabbix, Prometheus, Grafana), автоматизация (Ansible, Terraform/OpenTofu), виртуализация (Docker, Kubernetes), ОС (Debian Linux), СУБД (PostgreSQL, MySQL, Clickhouse), Message Broker (RabbitMQ), Reverse proxy/web server (Nginx), прочие инструменты (Jira, Confluence, Git). Требования: опыт работы с большинством из перечисленных технологий или готовность их изучить. Кандидат будет заниматься поддержкой и развитием CI/CD, мониторинга, автоматизацией административных задач, развитием инфраструктуры на Linux, внедрением контейнеризации, взаимодействием с разработчиками и тестировщиками. Компания предлагает: гибкий график работы, офис или гибридный формат, стабильность, ДМС, социальную поддержку, развитие, компенсацию обучения, корпоративы, тимбилдинги, удобные рабочие места, безлимитную еду.
Ключевые навыки: None
Тип занятости: Полная занятость


# Upload data

In [3]:
import pandas as pd

path = '/Users/richardgurtsiev/Desktop/projects/save/delete_2024/del/dl_skip/advisor/dataset/vac_dataset_2024-11-17.csv'
df = pd.read_csv(path)

In [7]:
# for idx, row in df.iterrows():
#     print(row)
#     print('*'*50)

# Milvus

In [1]:
from pymilvus import MilvusClient, model
from pymilvus.model.sparse.bm25.tokenizers import build_default_analyzer
from pymilvus.model.sparse import BM25EmbeddingFunction

client = MilvusClient(uri="http://localhost:19530", timeout=300)

#drop
# if client.has_collection(collection_name="demo_collection"):
#     client.drop_collection(collection_name="demo_collection")

# #create
client.create_collection(
    collection_name="demo_collection",
    dimension=768,
    primary_field_name="idx",
    vector_field_name="vectors",
    metric_type="L2",
    auto_id=True,
    description='База для проекта Advisor'
)


# embedding_fn = model.DefaultEmbeddingFunction()


# docs = [
#     "Artificial intelligence was founded as an academic discipline in 1956.",
#     "Alan Turing was the first person to conduct substantial research in AI.",
#     "Born in Maida Vale, London, Turing was raised in southern England.",
# ]

# vectors = embedding_fn.encode_documents(docs)

# The output vector has 768 dimensions, matching the collection that we just created.
# print("Dim:", embedding_fn.dim, vectors[0].shape) 


# entities = [
#     {"id": i, "vector": vectors[i], "text": docs[i], "subject": "history"}
#     for i in range(len(vectors))
# ]


# print("Data has", len(entities), "entities, each with fields: ", entities[0].keys())
# print("Vector dim:", len(entities[0]["vector"]))

/Users/richardgurtsiev/Desktop/projects/save/delete_2024/del/dl_skip/advisor/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
sentence_transformer_ef = model.SentenceTransformerEmbeddingFunction(
    model_name='all-MiniLM-L6-v2', # Specify the model name
    device='cpu' # Specify the device to use, e.g., 'cpu' or 'cuda:0'
)

docs = [
    "Artificial intelligence was founded as an academic discipline in 1956.",
    "Alan Turing was the first person to conduct substantial research in AI.",
    "Born in Maida Vale, London, Turing was raised in southern England.",
]

docs_embeddings = sentence_transformer_ef.encode_documents(docs)
print("Embeddings:", docs_embeddings)

AttributeError: The attribute 'SentenceTransformerEmbeddingFunction' is not found in 'pymilvus.model'. This might be due to an outdated version of 'milvus_model'. For upgrading to the latest version, use 'pip install milvus-model --upgrade'. For more information, please visit https://github.com/milvus-io/milvus-model.

In [3]:
queries = ["When was artificial intelligence founded", 
           "Where was Alan Turing born?"]

query_embeddings = sentence_transformer_ef.encode_queries(queries)

print("Embeddings:", query_embeddings)
print("Dim:", sentence_transformer_ef.dim, query_embeddings[0].shape)

NameError: name 'sentence_transformer_ef' is not defined

# Check

In [45]:
from sentence_transformers import SentenceTransformer

client = QdrantClient(url="http://localhost:6333")
collection_name = "advisor_db"

emb_model = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(emb_model)

text = """
Резюме: Backend-разработчик Java middle+
Опыт работы: От 3 до 6 лет
Описание: Компания занимается разработкой и внедрением инновационных решений для пассажирского транспорта, используя Java 8, 17, Spring framework, Spring Boot, JPA, Cloud, Security, AOP, Test Containers, Junit 4,5, Mockito, Kafka, PostgreSQL 14.5, PromQL, Grafana, Linux, Docker, Git, Tomcat, Netty, SSL, TLS, HTTP, HTTPS, Maven, Gradle, Gitlab CI/CD; требует уверенных знаний Java Core 8, 17, опыта работы с Spring Framework, Spring Boot, Spring JPA, Hibernate, JUnit, Mockito, Test Containers, SQL, инструментами разработки Gradle, Maven, Docker, Git, баг-трекерами (Taiga, Mantis и подобные), знание платежных систем и интеграции с платежными шлюзами, технологий смарт-карт, интеграции ККТ, ясный ум и умение разбираться и обучаться; кандидат будет заниматься бэк-энд разработкой для высоконагруженных сервисов, в том числе мобильных (автоматизация бизнеса, отрасли, платежи, приложения); компания предлагает аккредитованную IT-компанию, ДМС, внешнее обучение за счет компании, годовой бонус по результатам Performance Review, формат работы - офис, с возможностью перехода на гибридный график (г. Москва).
Ключевые навыки: None
Тип занятости: Полная занятость
График работы: Полный день
Местоположение: Москва
Профессиональные роли: Программист, разработчик
"""

def search(
        query: str,
        collection_name: str,
        topk: int = 10,
        filter_options: dict = None,
        score_threshold: float = None
    ):
    try:
        embedding = model.encode(query, normalize_embeddings=True)
        
        results = client.search(
            collection_name,
            embedding,
            limit=topk,
            query_filter=models.Filter(
                must=[
                    models.FieldCondition(key=k, match=models.MatchValue(value=v))
                    for k, v in filter_options.items()
                ]
            ) if filter_options else None,
            score_threshold=score_threshold
        )

        return results
    except Exception as error:
        raise Exception(f'Ошибка при поиске: {error}')


results = search(query=text,
       collection_name=collection_name,
       filter_options={"category": "backend", 'catalog':'cv'},
       score_threshold=0)

objects = []
vectors = [hit.vector for hit in results]
for idx, result in enumerate(results):
    print(result.payload['content'])
    objects.append(result.payload['content'])
    print('-'*50)
    print('\n')

Резюме: Backend разработчик
Желаемая зарплата: None
Опыт работы: Опыт работы 8 лет 4 месяца
Описание: None
Ключевые навыки: VueJS, Nuxt, Vue, JavaScript, MySQL, PHP, Laravel, ООП, API, Linux, PostgreSQL, Git, Symfony, Docker, Solid, RabbitMQ
Тип занятости: None
График работы: None
Знает языки: Русский — Родной, Английский — B1 — Средний
Образование: Неоконченное высшее образование 2018 Дальневосточный государственный университет путей сообщения, Хабаровск Автоматика
Местоположение: None
Профессиональные роли: Программист, разработчик
--------------------------------------------------


Резюме: Backend разработчик
Желаемая зарплата: None
Опыт работы: Опыт работы 1 год 8 месяцев
Описание: None
Ключевые навыки: Docker, PostgreSQL, SQL, Java, Spring Framework, Kotlin, ORACLE, Thrift, k8s, Backend
Тип занятости:  полная занятость, частичная занятость, проектная работа
График работы:  полный день, гибкий график, удаленная работа
Знает языки: Русский — Родной, Английский — B2 — Средне-продвин

In [49]:
# начальные данные

objects
print(text)


Резюме: Backend-разработчик Java middle+
Опыт работы: От 3 до 6 лет
Описание: Компания занимается разработкой и внедрением инновационных решений для пассажирского транспорта, используя Java 8, 17, Spring framework, Spring Boot, JPA, Cloud, Security, AOP, Test Containers, Junit 4,5, Mockito, Kafka, PostgreSQL 14.5, PromQL, Grafana, Linux, Docker, Git, Tomcat, Netty, SSL, TLS, HTTP, HTTPS, Maven, Gradle, Gitlab CI/CD; требует уверенных знаний Java Core 8, 17, опыта работы с Spring Framework, Spring Boot, Spring JPA, Hibernate, JUnit, Mockito, Test Containers, SQL, инструментами разработки Gradle, Maven, Docker, Git, баг-трекерами (Taiga, Mantis и подобные), знание платежных систем и интеграции с платежными шлюзами, технологий смарт-карт, интеграции ККТ, ясный ум и умение разбираться и обучаться; кандидат будет заниматься бэк-энд разработкой для высоконагруженных сервисов, в том числе мобильных (автоматизация бизнеса, отрасли, платежи, приложения); компания предлагает аккредитованную IT-

In [50]:
objects

['Резюме: Backend разработчик\nЖелаемая зарплата: None\nОпыт работы: Опыт работы 8 лет 4 месяца\nОписание: None\nКлючевые навыки: VueJS, Nuxt, Vue, JavaScript, MySQL, PHP, Laravel, ООП, API, Linux, PostgreSQL, Git, Symfony, Docker, Solid, RabbitMQ\nТип занятости: None\nГрафик работы: None\nЗнает языки: Русский — Родной, Английский — B1 — Средний\nОбразование: Неоконченное высшее образование 2018 Дальневосточный государственный университет путей сообщения, Хабаровск Автоматика\nМестоположение: None\nПрофессиональные роли: Программист, разработчик',
 'Резюме: Backend разработчик\nЖелаемая зарплата: None\nОпыт работы: Опыт работы 1 год 8 месяцев\nОписание: None\nКлючевые навыки: Docker, PostgreSQL, SQL, Java, Spring Framework, Kotlin, ORACLE, Thrift, k8s, Backend\nТип занятости:  полная занятость, частичная занятость, проектная работа\nГрафик работы:  полный день, гибкий график, удаленная работа\nЗнает языки: Русский — Родной, Английский — B2 — Средне-продвинутый\nОбразование: Высшее обра

# Hybrid search

In [23]:
from qdrant_client import QdrantClient, models
from fastembed.embedding import TextEmbedding
from fastembed.sparse.bm25 import Bm25
from fastembed.late_interaction import LateInteractionTextEmbedding
from datasets import load_dataset

dataset = load_dataset("BeIR/scifact", "corpus", split="corpus")


dense_embedding_model = TextEmbedding("sentence-transformers/all-MiniLM-L6-v2")
bm25_embedding_model = Bm25("Qdrant/bm25")
late_interaction_embedding_model = LateInteractionTextEmbedding("colbert-ir/colbertv2.0")

client = QdrantClient(url="http://localhost:6333", timeout=600)

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 51527.08it/s]


# Dense embeddings

In [24]:
from fastembed.embedding import TextEmbedding

dense_embedding_model = TextEmbedding("sentence-transformers/all-MiniLM-L6-v2")
dense_embeddings = list(dense_embedding_model.passage_embed(dataset["text"][0:1]))
len(dense_embeddings)

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 966.56it/s]


1

# Sparse embeddings

In [25]:
from fastembed.sparse.bm25 import Bm25

bm25_embedding_model = Bm25("Qdrant/bm25")
bm25_embeddings = list(bm25_embedding_model.passage_embed(dataset["text"][0:1]))
len(bm25_embeddings)

Fetching 29 files: 100%|██████████| 29/29 [00:00<00:00, 7876.37it/s]


1

# Late interaction embeddings

In [39]:
text = ['Hello, world']

1

In [41]:
from fastembed.late_interaction import LateInteractionTextEmbedding

late_interaction_embedding_model = LateInteractionTextEmbedding("colbert-ir/colbertv2.0")
late_interaction_embeddings = list(late_interaction_embedding_model.passage_embed(dataset["text"][0:1]))
len(late_interaction_embeddings)

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 35544.95it/s]


1

# Putting data in a Qdrant collection

In [27]:
from qdrant_client import QdrantClient, models

client = QdrantClient("http://localhost:6333", timeout=600)
client.create_collection(
    "scifact",
    vectors_config={
        "all-MiniLM-L6-v2": models.VectorParams(
            size=len(dense_embeddings[0]),
            distance=models.Distance.COSINE,
        ),
        "colbertv2.0": models.VectorParams(
            size=len(late_interaction_embeddings[0][0]),
            distance=models.Distance.COSINE,
            multivector_config=models.MultiVectorConfig(
                comparator=models.MultiVectorComparator.MAX_SIM,
            )
        ),
    },
    sparse_vectors_config={
        "bm25": models.SparseVectorParams(
            modifier=models.Modifier.IDF,
        )
    }
)

True

In [28]:
dataset = dataset.shuffle(seed=42).select(range(60))

In [29]:
import tqdm

batch_size = 4
for batch in tqdm.tqdm(dataset.iter(batch_size=batch_size), total=len(dataset) // batch_size):


    dense_embeddings = list(dense_embedding_model.passage_embed(batch["text"]))
    bm25_embeddings = list(bm25_embedding_model.passage_embed(batch["text"]))
    late_interaction_embeddings = list(late_interaction_embedding_model.passage_embed(batch["text"]))


    client.upload_points(
        "scifact",
        points=[
            models.PointStruct(
                id=int(batch["_id"][i]),
                vector={
                    "all-MiniLM-L6-v2": dense_embeddings[i].tolist(),
                    "bm25": bm25_embeddings[i].as_object(),
                    "colbertv2.0": late_interaction_embeddings[i].tolist(),
                },
                payload={
                    "_id": batch["_id"][i],
                    "title": batch["title"][i],
                    "text": batch["text"][i],
                }
            )
            for i, _ in enumerate(batch["_id"])
        ],
        # We send a lot of embeddings at once, so it's best to reduce the batch size.
        # Otherwise, we would have gigantic requests sent for each batch and we can
        # easily reach the maximum size of a single request.
        batch_size=batch_size,  
    )

100%|██████████| 15/15 [00:22<00:00,  1.49s/it]


In [31]:
client.get_collection('scifact'),client.count("scifact")

(CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, vectors_count=None, indexed_vectors_count=60, points_count=60, segments_count=8, config=CollectionConfig(params=CollectionParams(vectors={'all-MiniLM-L6-v2': VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), 'colbertv2.0': VectorParams(size=128, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=MultiVectorConfig(comparator=<MultiVectorComparator.MAX_SIM: 'max_sim'>))}, shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, on_disk_payload=True, sparse_vectors={'bm25': SparseVectorParams(index=None, modifier=<Modifier.IDF: 'idf'>)}), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_di

In [32]:
from fastembed.embedding import TextEmbedding
from fastembed.sparse.bm25 import Bm25
from fastembed.late_interaction import LateInteractionTextEmbedding

dense_embedding_model = TextEmbedding("sentence-transformers/all-MiniLM-L6-v2")
bm25_embedding_model = Bm25("Qdrant/bm25")
late_interaction_embedding_model = LateInteractionTextEmbedding("colbert-ir/colbertv2.0")

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 104335.92it/s]


In [34]:
query_text = "What is the impact of COVID-19 on the environment?"

result = client.query_points(
    "scifact",
    query=next(dense_embedding_model.query_embed(query_text)),
    using="all-MiniLM-L6-v2",
    limit=10,
    with_payload=False,
)

# Benchmarking

In [72]:
from qdrant_client import QdrantClient, models
import pandas as pd

client = QdrantClient("http://localhost:6333")
client.count("advisor_db")

CountResult(count=40)

In [73]:
df = pd.read_csv('test.csv', sep=',')
df.head()

,Unnamed: 0,catalog,category,content
0,0,vac,devops,Вакансия: DevOps-инженер\nОпыт работы: От 1 го...
1,1,vac,devops,Вакансия: DevOps - инженер\nОпыт работы: От 3 ...
2,2,vac,devops,Вакансия: DevOps (SQL направление)\nОпыт работ...
3,3,vac,devops,Вакансия: SRE/DevOps-инженер\nОпыт работы: От ...
4,4,vac,devops,Вакансия: DevOps\nОпыт работы: Более 6 лет\nОп...


In [191]:
from fastembed.embedding import TextEmbedding
from fastembed.sparse.bm25 import Bm25
from fastembed.late_interaction import LateInteractionTextEmbedding
from sentence_transformers import SentenceTransformer

dense_embedding_model_MiniLM = TextEmbedding("sentence-transformers/all-MiniLM-L6-v2")
dense_embedding_model_DEEPVK_USER = SentenceTransformer("deepvk/USER-bge-m3")
bm25_embedding_model = Bm25("Qdrant/bm25")
late_interaction_embedding_model = LateInteractionTextEmbedding("colbert-ir/colbertv2.0")

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 67650.06it/s]


In [144]:
from collections import defaultdict
from ranx import Qrels, Run, evaluate

query_text = """
Резюме: DevOps Опыт работы 5 месяца Описание: unknown Ключевые навыки: Управление разработкой, Azure DevOps, PowerShell, Jenkins, Git, Bash, Ansible, Nexus, Docker, Kubernetes, OpenShift, Groovy, Kafka, Nginx, TeamCity, PostgreSQL, SQL, Grafana, DevOps, Helm, Prometheus, CI/CD, Linux, YAML, Kotlin Тип занятости: unknown График работы: unknown Знание языков: Русский — Родной, Английский — A2 — Элементарный Образование: True
"""
ground_truth = 'cv-devops'


def evaluate_ranking(query: str, ground_truth: dict):

    targets = dict()
    similarity = dict()

    # result=client.query_points(
    #     "advisor_db",
    #     query=next(dense_embedding_model.query_embed(query_text)),
    #     using="all-MiniLM-L6-v2",
    #     limit=40,
    #     with_payload=True,
    # )

    result = client.query_points(
        "advisor_db",
        query=models.SparseVector(**next(bm25_embedding_model.query_embed(query)).as_object()),
        using="bm25",
        with_payload=True,
        limit=40,
    )
    print(result)

    points = result.points    

    for point in points:
        catalog = point.payload['catalog']
        category = point.payload['category']
        similar = (f'{catalog}-{category}')
        
        if similar == ground_truth:
            target = 1
        else:
            target = 0
        
        targets[f"doc_{point.id}"] = target
        similarity[f"doc_{point.id}"] = point.score
    
        
    qrels = {"query_1": targets}
    run = {"query_1": similarity}

    # print(qrels)
    # print(run)
    
    # for i in zip(qrels['query_1'], run['query_1']):
    #     print(i)
    
    results = evaluate(qrels, run, metrics=["ndcg", "precision@10", "map@10"])
    
    return results

evaluate_ranking(query_text, ground_truth=ground_truth)

points=[ScoredPoint(id=25, version=26, score=81.776306, payload={'catalog': 'cv', 'category': 'devops', 'content': 'Резюме: DevOps\nОпыт работы 14 лет 3 месяца\nОписание: unknown\nКлючевые навыки: Управление разработкой, Azure DevOps, PowerShell, Jenkins, Git, Bash, Ansible, Nexus, Docker, Kubernetes, OpenShift, Groovy, Kafka, Nginx, TeamCity, PostgreSQL, SQL, Grafana, DevOps, Helm, Prometheus, CI/CD, Linux, YAML, Kotlin\nТип занятости: unknown\nГрафик работы: unknown\nЗнание языков: Русский — Родной, Английский — A2 — Элементарный\nОбразование: True'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=22, version=25, score=29.30199, payload={'catalog': 'cv', 'category': 'devops', 'content': 'Резюме: DevOps\nОпыт работы 10 лет 1 месяц\nОписание: unknown\nКлючевые навыки: Percona, CICD, Ansible, Bash, Restapi, Jenkins, Zabbix, Python, OpenShift, Kafka, Bitbucket, ELK, Vault, Gradle, Groovy, Apache Maven, Spring Framework, Redhat, Prometheus, IaaS, PaaS, SaaS, Английский язы

{'ndcg': 0.9259119060300831, 'precision@10': 0.7, 'map@10': 0.639047619047619}

In [197]:
from collections import defaultdict
from ranx import Qrels, Run, evaluate

query_text = """
Резюме: DevOps Опыт работы 5 месяца Описание: unknown Ключевые навыки: Управление разработкой, Azure DevOps, PowerShell, Jenkins, Git, Bash, Ansible, Nexus, Docker, Kubernetes, OpenShift, Groovy, Kafka, Nginx, TeamCity, PostgreSQL, SQL, Grafana, DevOps, Helm, Prometheus, CI/CD, Linux, YAML, Kotlin Тип занятости: unknown График работы: unknown Знание языков: Русский — Родной, Английский — A2 — Элементарный Образование: True
"""
ground_truth = "cv-devops"


def hybrif_search(query:str, type:str, collection_name:str, limit:int):
        if type == 'min':
            results = client.query_points(
                collection_name,
                query=next(dense_embedding_model_MiniLM.query_embed(query)),
                using="all-MiniLM-L6-v2",
                limit=limit,
                with_payload=True,
            )
        elif type == 'bm25':
            results = client.query_points(
                collection_name,
                query=models.SparseVector(**next(bm25_embedding_model.query_embed(query)).as_object()),
                using="bm25",
                with_payload=True,
                limit=limit,
            )
        elif type == 'late':
            results = client.query_points(
                collection_name,
                query=next(late_interaction_embedding_model.query_embed(query)),
                using="colbertv2.0",
                with_payload=True,
                limit=limit,
            )
        elif type == 'vk':
            results = client.query_points(
                collection_name,
                query=dense_embedding_model_DEEPVK_USER.encode(query, normalize_embeddings=True),
                using="deepvk/USER-bge-m3",
                with_payload=True,
                limit=limit,
            )            
        
        # hybrid1 --> min, bm25,  
        elif type == 'hybrid':
            prefetch = [
                models.Prefetch(
                    query=next(dense_embedding_model_MiniLM.query_embed(query)),
                    using="all-MiniLM-L6-v2",
                    limit=limit,
                ),
                models.Prefetch(
                    query=models.SparseVector(**next(bm25_embedding_model.query_embed(query)).as_object()),
                    using="bm25",
                    limit=limit,
                ),
            ]
            results = client.query_points(
                collection_name,
                prefetch=prefetch,
                query=models.FusionQuery(
                    fusion=models.Fusion.RRF,
                ),
                with_payload=True,
                limit=limit,
            )            
        # hybrid1 --> min, bm25, 
        elif type == 'all':
            prefetch = [
                models.Prefetch(
                    query=next(dense_embedding_model_MiniLM.query_embed(query)),
                    using="all-MiniLM-L6-v2",
                    limit=limit,
                ),
                models.Prefetch(
                    query=models.SparseVector(**next(bm25_embedding_model.query_embed(query)).as_object()),
                    using="bm25",
                    limit=limit,
                ),
                models.Prefetch(
                    query=next(late_interaction_embedding_model.query_embed(query)),
                    using="colbertv2.0",
                    limit=limit,
                ),
                models.Prefetch(
                    query=dense_embedding_model_DEEPVK_USER.encode(query, normalize_embeddings=True),
                    using="deepvk/USER-bge-m3",
                    limit=limit,
                ),
            ]
            results = client.query_points(
                collection_name,
                prefetch=prefetch,
                query=models.FusionQuery(
                    fusion=models.Fusion.RRF,
                ),
                with_payload=True,
                limit=limit,
            )
        # late all
        elif type == 'all_late':

            prefetch = [
                models.Prefetch(
                    query=next(dense_embedding_model_MiniLM.query_embed(query)),
                    using="all-MiniLM-L6-v2",
                    limit=limit,
                ),
                models.Prefetch(
                    query=models.SparseVector(**next(bm25_embedding_model.query_embed(query)).as_object()),
                    using="bm25",
                    limit=limit,
                ),
            ]
            results = client.query_points(
                collection_name,
                prefetch=prefetch,
                query=next(late_interaction_embedding_model.query_embed(query)),
                using="colbertv2.0",
                with_payload=True,
                limit=limit,
            )            

        elif type == 'multy':
            results = client.query_points(
                    collection_name,
                    prefetch=[
                        models.Prefetch(
                            prefetch=[
                                models.Prefetch(
                                    query=next(dense_embedding_model_MiniLM.query_embed(query)),
                                    using="all-MiniLM-L6-v2",
                                    limit=40,
                                )
                            ],
                            query=models.SparseVector(**next(bm25_embedding_model.query_embed(query)).as_object()),
                            using="bm25",
                            limit=10,
                        ),
                    ],
                    query=next(late_interaction_embedding_model.query_embed(query)),
                    using="colbertv2.0",
                    with_payload=True,
                    limit=limit,
                )            

        
        return results

def evaluate_ranking(query: str, ground_truth_label: str):
    # try:
        results = hybrif_search(query, 'vk', 'test', 10)
        
        points = results.points
        if not points:
            raise ValueError("No points returned from the query.")

        targets = {}
        for point in points:
            catalog = point.payload['catalog']
            category = point.payload['category']
            similar = f"{catalog}-{category}"

            target = 1 if similar == ground_truth_label else 0
            targets[f"doc_{point.id}"] = target

        qrels = {"query_1": targets}
        run = {"query_1": {f"doc_{point.id}": point.score for point in points}}

        results = evaluate(
            qrels, 
            run, 
            metrics=["ndcg", "precision@10", "map@10"], 
            make_comparable=True
        )
        return results

    # except Exception as e:
    #     print(f"Error during evaluation: {e}")
    #     return None

metrics = evaluate_ranking(query_text, ground_truth_label=ground_truth)
if metrics:
    print("Metrics:", metrics)
else:
    print("Evaluation failed.")

Metrics: {'ndcg': 1.0, 'precision@10': 1.0, 'map@10': 1.0}


In [55]:
run_dict = {}
for query_idx, query in enumerate(queries):
    query_id = str(query["_id"])
    
    query_vector = sparse_vectors[query_idx]
    
    results = client.query_points(
        "scifact",
        query=models.SparseVector(**query_vector.as_object()),
        using="bm25",
        with_payload=False,
        limit=10,
    )
    
    run_dict[query_id] = {
        str(point.id): point.score
        for point in results.points
    }
    
bm25_run = Run(run_dict, name="bm25")
evaluate(qrels, bm25_run, metrics=["precision@10", "mrr@10"], make_comparable=Truу

In [56]:
# from ranx import Qrels
# from collections import defaultdict

# qrels_dict = defaultdict(dict)
# for entry in query_qrels:
#     query_id = str(entry["query-id"])
#     doc_id = str(entry["corpus-id"])
#     qrels_dict[query_id][doc_id] = entry["score"]

# qrels = Qrels(qrels_dict, name="scifact")
# qrels

DictType[unicode_type,DictType[[unichr x 9],int64]<iv=None>]<iv=None>({0: {31715818: 1}, 10: {32587939: 1}, 1000: {16472469: 1}, 1001: {5702790: 1}, 1002: {13639330: 1}, 1003: {14332945: 1, 4319844: 1, 4899981: 1}, 1004: {301838: 1, 2734421: 1, 3952288: 1}, 1005: {301838: 1, 2734421: 1, 3952288: 1}, 1006: {4926049: 1}, 1008: {2547636: 1}, 1009: {1982286: 1}, 1011: {9745001: 1}, 1015: {6277638: 1}, 1016: {6277638: 1}, 1018: {11603066: 1}, 1023: {16927286: 1}, 1025: {32408470: 1}, 1026: {3113630: 1}, 1027: {3113630: 1}, 1028: {13923140: 1, 11899391: 1}, 1030: {6441369: 1}, 1031: {12486491: 1}, 1032: {6836086: 1}, 1033: {6836086: 1}, 1034: {4547102: 1}, 1035: {4547102: 1}, 1036: {4547102: 1}, 1037: {16287725: 1}, 1038: {16287725: 1}, 104: {40164383: 1}, 1040: {25254425: 1, 16626264: 1}, 1042: {17421851: 1}, 1043: {17671145: 1}, 1044: {22500262: 1}, 1045: {22500262: 1}, 1046: {418246: 1, 4324278: 1, 16712164: 1}, 1047: {14706752: 1}, 1048: {12486491: 1}, 105: {36606083: 1}, 1050: {19878070

In [ ]:
import tqdm 

dense_vectors, sparse_vectors, late_vectors = [], [], []
for query in tqdm.tqdm(queries):
    dense_query_vector = next(dense_embedding_model.query_embed(query["text"]))
    sparse_query_vector = next(bm25_embedding_model.query_embed(query["text"]))
    late_query_vector = next(late_interaction_embedding_model.query_embed(query["text"]))

    dense_vectors.append(dense_query_vector)
    sparse_vectors.append(sparse_query_vector)
    late_vectors.append(late_query_vector)

# Search

In [256]:
from typing import List

def create_prefetch_from_models(query: str, models_list: List[str], limit: int) -> List[models.Prefetch]:
    """
    Создает список Prefetch для гибридного поиска на основе списка моделей.
    
    Args:
        query (str): Текст запроса.
        models_list (List[str]): Список строковых названий моделей.
        limit (int): Максимальное количество результатов на запрос.
    
    Returns:
        List[models.Prefetch]: Список объектов Prefetch.
    """
    prefetch = []
    
    for model_name in models_list:
        if model_name == "all-MiniLM-L6-v2":
            prefetch.append(
                models.Prefetch(
                    query=next(dense_embedding_model_MiniLM.query_embed(query)),
                    using="all-MiniLM-L6-v2",
                    limit=limit,
                )
            )
        elif model_name == "bm25":
            prefetch.append(
                models.Prefetch(
                    query=models.SparseVector(**next(bm25_embedding_model.query_embed(query)).as_object()),
                    using="bm25",
                    limit=limit,
                )
            )
        elif model_name == "colbertv2.0":
            prefetch.append(
                models.Prefetch(
                    query=next(late_interaction_embedding_model.query_embed(query)),
                    using="colbertv2.0",
                    limit=limit,
                )
            )
        elif model_name == "deepvk/USER-bge-m3":
            prefetch.append(
                models.Prefetch(
                    query=dense_embedding_model_DEEPVK_USER.encode(query, normalize_embeddings=True),
                    using="deepvk/USER-bge-m3",
                    limit=limit,
                )
            )
        else:
            raise ValueError(f"Неизвестная модель: {model_name}")
    
    return prefetch


def hybrid_query_dynamic(collection_name: str, query: str, models_list: List[str], limit: int):
    """
    Выполняет гибридный запрос с использованием динамического создания Prefetch.
    
    Args:
        collection_name (str): Название коллекции в базе.
        query (str): Текст запроса.
        models_list (List[str]): Список названий моделей для Prefetch.
        limit (int): Максимальное количество результатов.
    
    Returns:
        Any: Результаты запроса.
    """
    prefetch = create_prefetch_from_models(query=query, models_list=models_list, limit=limit)
    
    results = client.query_points(
        collection_name=collection_name,
        prefetch=prefetch,
        query=models.FusionQuery(
            fusion=models.Fusion.RRF,
        ),
        with_payload=True,
        limit=limit,
    )
    return results

In [325]:

models_list = ["all-MiniLM-L6-v2", "bm25", "deepvk/USER-bge-m3", "colbertv2.0"]
models_list = ["all-MiniLM-L6-v2", "bm25", "deepvk/USER-bge-m3"]
models_list = ["deepvk/USER-bge-m3", "bm25", "colbertv2.0"]
models_list = ["bm25", "deepvk/USER-bge-m3", "colbertv2.0"]
models_list = ["all-MiniLM-L6-v2", "bm25"] # 0.64
models_list = ["all-MiniLM-L6-v2", "bm25", "colbertv2.0"] # 0.75

limit = 10

query_text = """
Резюме: SRE/DevOps-инженер 
Опыт работы: От 3 до 6 лет 
Описание: unknown
Тип занятости: Полная занятость 
График работы: Удаленная работа 
Знание языков: unknown 
Образование: False
"""

ground_truth = "cv-devops"

result = hybrid_query_dynamic(collection_name='test', query=query_text, models_list=models_list, limit=10)

In [327]:
print(query_text)
for i in result.points:
    print(i.payload['content'])
    print()


Резюме: SRE/DevOps-инженер 
Опыт работы: От 3 до 6 лет 
Описание: unknown
Тип занятости: Полная занятость 
График работы: Удаленная работа 
Знание языков: unknown 
Образование: False

Резюме: DevOps
Опыт работы 14 лет 4 месяца
Описание: unknown
Ключевые навыки: Linux, Docker
Тип занятости: unknown
График работы: unknown
Знание языков: Русский — Родной, Английский — B1 — Средний
Образование: True

Вакансия: SRE/DevOps-инженер
Опыт работы: От 3 до 6 лет
Описание: Описание: Компания занимается автоматизацией маркетинга, разрабатывает HighLoad SaaS сервисы, использует технологии Linux, Kubernetes, Docker, Postgres, ClickHouse, Redis, memcached, RabbitMQ, Grafana, PromQL, PHP, Golang, Java/Kotlin, Bash, Ansible, Terraform, Apache Airflow, TypeScript, Python, NodeJS, Deckhouse Kubernetes Platform, Graylog, Prometheus, GitLab, Active Directory, MS RDS, werf, требует опыт администрирования серверов Linux, эксплуатации приложений в системах оркестрации и контейнеризации, работы с СУБД и NoSQL 

In [326]:
def evaluate_ranking(ground_truth:str, result, limit=limit):
    try:        
        points = result.points
        if not points:
            raise ValueError("No points returned from the query.")

        targets = {}
        for point in points:
            catalog = point.payload['catalog']
            category = point.payload['category']
            similar = f"{catalog}-{category}"

            target = 1 if similar == ground_truth else 0
            targets[f"doc_{point.id}"] = target

        qrels = {"query_1": targets}
        run = {"query_1": {f"doc_{point.id}": point.score for point in points}}

        ranking_assessment = evaluate(
            qrels, 
            run, 
            metrics=["ndcg", f"precision@{limit}", f"map@{limit}"], 
            make_comparable=True
        )
        return ranking_assessment

    except Exception as e:
        print(f"Error during evaluation: {e}")
        return None

metrics = evaluate_ranking(ground_truth=ground_truth, result=result, limit=10)
if metrics:
    print("Metrics:", metrics)
else:
    print("Evaluation failed.")

Metrics: {'ndcg': 0.8854598815714874, 'precision@10': 0.3, 'map@10': 0.7555555555555555}


In [54]:
from sentence_transformers import SentenceTransformer, util


model = SentenceTransformer("deepvk/USER-bge-m3")


target="Резюме: DevOps Опыт работы 5 месяца Описание: unknown Ключевые навыки: Управление разработкой, Azure DevOps, PowerShell, Jenkins, Git, Bash, Ansible, Nexus, Docker, Kubernetes, OpenShift, Groovy, Kafka, Nginx, TeamCity, PostgreSQL, SQL, Grafana, DevOps, Helm, Prometheus, CI/CD, Linux, YAML, Kotlin Тип занятости: unknown График работы: unknown Знание языков: Русский — Родной, Английский — A2 — Элементарный Образование: True"

docs = ["Резюме: DevOps Опыт работы 5,1 месяца Описание: unknown Ключевые навыки: Cisco, Английский язык, Чешский язык, Водительское удостоверение категории B, Грамотная речь, Работа с большим объемом информации, Работа в команде, Пользователь ПК, Грамотность, ICND1, Java Fundamentals, Linux, Docker, Kubernetes, Деловое общение, Деловая переписка, Локальные сети, AutoCAD, Компас, Информационная безопасность, Jenkins Тип занятости: unknown График работы: unknown Знание языков: Русский — Родной, Английский — B1 — Средний, Чешский — B2 — Средне-продвинутый Образование: True",
"Резюме: DevOps Опыт работы 10,2 месяца Описание: unknown Ключевые навыки: PHP, JavaScript, MySQL, jQuery, Git, HTML5, 1С-Битрикс, PostgreSQL, ООП, Yii, Bootstrap, MVC, MongoDB, CI/CD, автотесты Python, Ansible, Terraform, Zabbix, Docker, Bash, Nginx, Apache HTTP Server, mssql, Docker-compose, Gitlab, SOAP, Linux, Prometheus Тип занятости: полная занятость, частичная занятость, проектная работа График работы: гибкий график, удаленная работа Знание языков: Русский — Родной, Английский — B2 — Средне-продвинутый Образование: True",
"Резюме: DevOps Опыт работы 10,1 месяц Описание: unknown Ключевые навыки: Percona, CICD, Ansible, Bash, Restapi, Jenkins, Zabbix, Python, OpenShift, Kafka, Bitbucket, ELK, Vault, Gradle, Groovy, Apache Maven, Spring Framework, Redhat, Prometheus, IaaS, PaaS, SaaS, Английский язык, SQL, Gitlab, Yandex Cloud, Docker Compose, GitLab CI, Helm Тип занятости: unknown График работы: unknown Знание языков: Русский — Родной, Английский — B2 — Средне-продвинутый Образование: True",
"Резюме: DevOps Опыт работы 14,4 месяца Описание: unknown Ключевые навыки: Linux, Docker Тип занятости: unknown График работы: unknown Знание языков: Русский — Родной, Английский — B1 — Средний Образование: True",
"Резюме: DevOps engineer WОпыт работы 5 месяца Описание: unknown Ключевые навыки: Linux, Docker, Bash, Ansible, SQL, Jenkins, Unix, DevOps, OpenShift, Git, Gitlab, Администрирование, Kubernetes Тип занятости: unknown График работы: unknown Знание языков: Russian — Native, English — B1 — Intermediate Образование: True"]


embed_list = []

target_embed = model.encode(target, normalize_embeddings=True)
doc_embeddings = model.encode(docs, normalize_embeddings=True)

similarities = util.cos_sim(target_embed, doc_embeddings)[0]

# Создание списка с результатами
results = list(zip(docs, similarities.tolist()))

# Сортировка по убыванию сходства
sorted_results = sorted(results, key=lambda x: x[1], reverse=True)

# Вывод результатов
print("Резюме, отсортированные по сходству с целевым резюме:")
for idx, (doc, similarity) in enumerate(sorted_results, 1):
    print(f"{idx}. Сходство: {similarity:.4f}\n{doc}\n")


Резюме, отсортированные по сходству с целевым резюме:
1. Сходство: 0.9098
Резюме: DevOps engineer WОпыт работы 5 месяца Описание: unknown Ключевые навыки: Linux, Docker, Bash, Ansible, SQL, Jenkins, Unix, DevOps, OpenShift, Git, Gitlab, Администрирование, Kubernetes Тип занятости: unknown График работы: unknown Знание языков: Russian — Native, English — B1 — Intermediate Образование: True

2. Сходство: 0.8962
Резюме: DevOps Опыт работы 10,1 месяц Описание: unknown Ключевые навыки: Percona, CICD, Ansible, Bash, Restapi, Jenkins, Zabbix, Python, OpenShift, Kafka, Bitbucket, ELK, Vault, Gradle, Groovy, Apache Maven, Spring Framework, Redhat, Prometheus, IaaS, PaaS, SaaS, Английский язык, SQL, Gitlab, Yandex Cloud, Docker Compose, GitLab CI, Helm Тип занятости: unknown График работы: unknown Знание языков: Русский — Родной, Английский — B2 — Средне-продвинутый Образование: True

3. Сходство: 0.8823
Резюме: DevOps Опыт работы 10,2 месяца Описание: unknown Ключевые навыки: PHP, JavaScript, M